In [1]:
# Starter kit

import pandas as pd
df_train = pd.read_csv('train_data.csv')
df_test = pd.read_csv('test_data.csv')
df_train.head()

subtask1 ='dif_NT-MEV'
subtask2='loc-MEV'
subtask3='status_admitere'

df_train = df_train[['id', 'NT', 'MEV', 'MATE', 'MGIM', 'status_admitere']]
df_test = df_test[['id', 'NT', 'MEV', 'MATE', 'MGIM']]

df_test.head()

,id,NT,MEV,MATE,MGIM
0,289,9.52,6.24,6.97,5.56
1,312,8.47,9.47,7.56,7.91
2,423,7.59,8.28,5.18,9.37
3,348,6.70,7.44,8.99,8.17
4,415,1.82,6.71,8.67,8.34


In [2]:
features = ['NT', 'MEV', 'MATE', 'MGIM']
target = 'status_admitere'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_X = pd.DataFrame(scaler.fit_transform(X), columns=features)
scaled_X_final = pd.DataFrame(scaler.fit_transform(X_final), columns=features)

scaled_X

,NT,MEV,MATE,MGIM
0,-0.132210,0.626925,-0.947687,-0.524319
1,1.710842,-0.662863,0.988034,0.545522
2,0.397818,0.464634,-1.534054,0.673053
3,-1.561678,1.199215,1.051616,-1.417034
4,-1.372956,-1.175362,0.776094,0.531352
...,...,...,...,...
265,-1.609863,1.549423,0.959775,-1.792541
266,-1.561678,0.464634,-0.516742,-0.113387
267,-1.047712,-0.287031,-0.566194,0.538437
268,1.626519,-0.227239,-0.191767,1.147750


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.naive_bayes import GaussianNB

X_train, X_validation, y_train, y_validation = train_test_split(scaled_X, y, test_size=0.2, random_state=0)

model = GaussianNB()
model = model.fit(X_train, y_train)
y_pred = model.predict(X_validation)
print(accuracy_score(y_pred, y_validation))

1.0


In [4]:
mevs = pd.DataFrame({'id': df_test.id, 'MEV': df_test.MEV})
mevs = mevs.sort_values(by='MEV', ascending=False).reset_index(drop=True)

mevs

,id,MEV
0,434,10.00
1,303,9.99
2,394,9.95
3,318,9.95
4,436,9.91
...,...,...
175,270,6.13
176,405,6.12
177,309,6.12
178,424,6.12


In [5]:
df_test[subtask3] = model.predict(scaled_X_final)
df_test[subtask1] = 0
df_test[subtask2] = 0

def get_position(id):
    for i in range(len(mevs)):
        if mevs.iloc[i].id == id:
            return i+1

def subtasks(row):
    diferenta = row.NT - row.MEV
    diferenta = int(diferenta*100)/100
    row[subtask1] = diferenta
    row[subtask2] = get_position(row.id)
    return row

df_test = df_test.apply(subtasks, axis='columns')

df_1 = pd.DataFrame({'subtaskID': 1, 'datapointID': df_test.id, 'answer': df_test[subtask1]})
df_2 = pd.DataFrame({'subtaskID': 2, 'datapointID': df_test.id, 'answer': df_test[subtask2]})
df_3 = pd.DataFrame({'subtaskID': 3, 'datapointID': df_test.id, 'answer': df_test[subtask3]})

submission = pd.concat([df_1, df_2, df_3])
submission.to_csv('output.csv', index=False)

df_2.head()

,subtaskID,datapointID,answer
0,2,289.0,168.0
1,2,312.0,31.0
2,2,423.0,88.0
3,2,348.0,122.0
4,2,415.0,150.0
